In [20]:
import networkx as nx
import itertools

from bp.world import random_grid_world, Scenario

demands = {
    ((1, 0), (1, 3)): 6,
    ((0, 0), (2, 3)): 6,
}

world = random_grid_world(
    rows=4,
    cols=4,
    demands=demands,
    seed=0,
)
G = world.network.graph
world.network.bpr_beta = 1
nominal = Scenario.from_world("nominal", world)

print(f"{world.total_population=}")
for od, n_k in demands.items():
    print(f"Demand for {od}: {n_k}")

world.total_population=12
Demand for ((1, 0), (1, 3)): 6
Demand for ((0, 0), (2, 3)): 6


In [21]:
accident_congestion = 10
target_edge = ((1, 1), (1, 2))

travel_time = dict(nominal.travel_time)
travel_time[target_edge] += accident_congestion

accident = Scenario(
    name="accident",
    travel_time=travel_time,
    discomfort=nominal.discomfort,
    hazard=nominal.hazard,
    cost=nominal.cost,
    emissions=nominal.emissions,
    policing=nominal.policing
)

scenarios = {
    "nominal": (nominal, .8),
    "accident": (accident, .2),
}

assert all(prior >= 0 for _, prior in scenarios.values()), "invalid prior distribution"
assert sum(prior for _, prior in scenarios.values()) == 1, "invalid prior distribution"

In [22]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("asymmetric dictator (anonymous)")
model.setParam("OutputFlag", 0)

V = world.ordered_nodes
A = world.ordered_arcs
I = world.I
N = world.individuals

n = world.total_population
t = world.network.travel_time
c = world.network.capacity
alpha = world.network.bpr_alpha
beta = world.network.bpr_beta

In [23]:
from collections.abc import Mapping, Sequence

from bp.world import Arc, Node

def edge_path(path: Sequence[Node]) -> Sequence[Arc]:
    return list(itertools.pairwise(path))

In [24]:
# reminder: beta=1 is fixed
# tau[omega, a, k] := average cost for k players on arc a under state omega
tau = {}
for scenario_name, (omega, _) in scenarios.items():
    for a in A:
        for k in range(n + 1):
            tau[scenario_name, a, k] = omega.travel_time[a] * (1 + alpha * ((k - 1) / c[a]) ** beta)

In [ ]:
# decision: r[omega, i, a] := 1 iff player i uses arc a under scenario omega
r = {
    (scenario_name, i, a): model.addVar(vtype=GRB.BINARY, name=f"r_{scenario_name}_{i.id}_{a}")
    for i in N
    for a in A
    for scenario_name in scenarios
}

# decision: x[omega, a] := total flow on arc a under scenario omega
x = {
    (scenario_name, a): model.addVar(vtype=GRB.INTEGER, lb=0, ub=n, name=f"x_{scenario_name}_{a}")
    for a in A
    for scenario_name in scenarios
}

# constraint: flow conservation
for scenario_name in scenarios:
    for i in N:
        for v in V:
            flow_out = gp.quicksum(r[scenario_name, i, a] for a in A if a[0] == v)
            flow_in = gp.quicksum(r[scenario_name, i, a] for a in A if a[1] == v)
            flow = 1 if v == i.demand.origin else (-1 if v == i.demand.destination else 0)

            model.addConstr(flow_out - flow_in == flow, name=f"flow_{scenario_name}_{i.id}_{v}")

# constraint: x[omega, a] = \sum_{i \in N} r[omega, i, a]
for scenario_name in scenarios:
    for a in A:
        model.addConstr(
            x[scenario_name, a] == gp.quicksum(r[scenario_name, i, a] for i in N),
            name=f"x_{a}"
        )

# NOTE: erm not actually incorporating mu
# objective: \min \sum_{\omega \in \Omega} \mu_\omega \sum_{a \in A} x[\omega, a] \cdot tau[\omega, a, x[a]]
for scenario_name, (_, mu) in scenarios.items():
    for a in A:
        model.setPWLObj(
            x[scenario_name, a],
            list(range(n + 1)),
            [mu * k * tau[scenario_name, a, k] for k in range(n + 1)]
        )

In [26]:
model.optimize()
assert model.Status == GRB.OPTIMAL, f"Optimization failed: {model.Status}"

In [27]:
print(f"{model.ObjVal=:.2f}")
print(f"{model.Runtime=:.3f}s")

model.ObjVal=350.21
model.Runtime=0.014s
